# Smart City: Does Weather Affect Traffic?

**Formal speed hypotheses**  
H0: Rainfall has no effect on mean traffic speed; mean speed is equal in rainy and dry conditions after controlling for time of day and weekday.  
H1: Rainfall reduces traffic speed and is therefore associated with increased congestion.

**Formal volume hypotheses**  
H0-volume: Rainfall does not change mean traffic volume.  
H1-volume: Rainfall changes mean traffic volume.

Use alpha = 0.05. The speed hypothesis is decided from the rainfall coefficient in the controlled OLS model with HC3 robust standard errors; the rain-versus-dry Welch test is reported separately as an unadjusted comparison.

**Data assumption:** no source CSV was present when this project was initialized, so the pipeline generates reproducible synthetic observations (seed 42). All resulting city-level values are illustrative, not real-world measurements.

## Scope and cleaning decisions

The pipeline uses a supplied `traffic*.csv` or `smart_city*.csv` in the project root or `data/` when present; otherwise, it creates 15 roads x 90 days x 24 hours. Cleaning removes exact duplicate rows, drops unparseable timestamps, treats out-of-domain speed and negative volume/rain as sensor errors, and imputes missing values using road/time or weather-conditioned medians. Time of day and weekday are always recalculated from the timestamp. Domain-valid IQR outliers remain in the analysis and are counted in the quality outputs.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.traffic_weather_analysis import (
    load_and_clean,
    exploratory_analysis,
    statistical_analysis,
    train_and_evaluate_models,
    write_report,
)

## 1. Validate and clean

The before/after audit reports row counts, duplicates, missingness, and impossible readings. Inspect the final dtypes and documented treatment decisions before interpreting results.

In [2]:
data, quality = load_and_clean()
quality_summary = pd.DataFrame([quality["before"], quality["after"]]).set_index("stage").T
display(quality_summary)
print("Source:", quality["source"], "| synthetic:", quality["synthetic"])
print("Cleaning decisions:")
for decision in quality["decisions"]:
    print("-", decision)
print("Cleaned data sample:")
display(data.head())

DATA QUALITY SUMMARY (before / after)
stage                      before  after
rows                        32428  32400
duplicate_rows                 28      0
missing_Timestamp               0      0
missing_Traffic_Speed         194      0
missing_Traffic_Volume        193      0
missing_Rainfall              194      0
missing_Weather_Condition       0      0
missing_Time_of_Day             0      0
missing_Weekday                 0      0
invalid_speed                  97      0
invalid_volume                 97      0
invalid_rainfall               97      0

Cleaned dtypes
Timestamp            datetime64[us]
Road_ID                         str
Traffic_Speed               float64
Traffic_Volume              float64
Rainfall                    float64
Weather_Condition            string
Time_of_Day                     str
Weekday                         str
is_weekend                     bool
hour                          int32
Dropped unparseable timestamps: 0


stage,before,after
rows,32428,32400
duplicate_rows,28,0
missing_Timestamp,0,0
missing_Traffic_Speed,194,0
missing_Traffic_Volume,193,0
missing_Rainfall,194,0
missing_Weather_Condition,0,0
missing_Time_of_Day,0,0
missing_Weekday,0,0
invalid_speed,97,0


Source: data\traffic_raw.csv | synthetic: True
Cleaning decisions:
- Exact duplicate rows were dropped before parsing to prevent repeated sensor readings from weighting estimates.
- Unparseable timestamps were dropped because time is required for all temporal controls; mixed formats were parsed with format='mixed'.
- Traffic speeds outside 5-90 km/h, negative volumes, and negative rainfall were set missing as physically/domain-impossible glitches.
- Missing or impossible speed and volume were median-imputed within Road_ID x Time_of_Day, then by road; valid domain-range extremes were retained.
- Missing rainfall was set to zero for dry labels and to the weather-condition median for rain labels; weather labels were reconciled to rainfall thresholds.
- Time_of_Day and Weekday were always recomputed from Timestamp to resolve stale or contradictory labels.
- IQR outliers were flagged in summaries, not capped or removed: high but valid counts/rainfall can be genuine events and capping would 

,Timestamp,Road_ID,Traffic_Speed,Traffic_Volume,Rainfall,Weather_Condition,Time_of_Day,Weekday,is_weekend,hour
0,2025-01-06,R01,65.81,776.8,0.0,Cloudy,Night,Monday,False,0
1,2025-01-06,R02,65.72,991.4,0.0,Cloudy,Night,Monday,False,0
2,2025-01-06,R03,68.49,450.4,0.0,Cloudy,Night,Monday,False,0
3,2025-01-06,R04,60.96,568.9,0.0,Cloudy,Night,Monday,False,0
4,2025-01-06,R05,74.25,755.4,0.0,Cloudy,Night,Monday,False,0


## 2. Exploratory analysis

Descriptive summaries cover the full dataset and each requested weather, road, time-of-day, and weekday grouping. Numeric correlations and IQR outlier counts are saved under `outputs/`.

In [3]:
eda = exploratory_analysis(data)
print("Overall statistics")
display(eda["tables"]["overall"].round(3))
print("Weather statistics")
display(eda["tables"]["weather"].round(3))
print("Road statistics")
display(eda["tables"]["road"].round(3))
print("Time-of-day statistics")
display(eda["tables"]["time_of_day"].round(3))
print("Weekday statistics")
display(eda["tables"]["weekday"].round(3))
display(eda["correlations"].round(3))
display(eda["iqr"].round(3))

Overall descriptive statistics
       measure    mean  median     std   min     max   skew   count
 Traffic_Speed  61.892   62.55  10.411 19.87   90.00 -0.282 32400.0
Traffic_Volume 642.049  601.60 228.698 84.80 1585.60  0.914 32400.0
      Rainfall   0.682    0.00   2.055  0.00   20.76  4.381 32400.0

Weather-group descriptive statistics
Weather_Condition        measure    mean  median     std    min     max   skew  count
            Clear  Traffic_Speed  63.015  63.830  10.029  29.21   90.00 -0.255  17913
           Cloudy  Traffic_Speed  62.560  63.340  10.138  31.63   90.00 -0.220   6227
              Fog  Traffic_Speed  63.625  64.505   9.768  33.35   87.11 -0.363   1614
       Heavy Rain  Traffic_Speed  54.485  54.740  11.538  19.87   82.66 -0.189   1965
       Light Rain  Traffic_Speed  59.221  59.300  10.162  25.05   85.64 -0.159   4681
            Clear Traffic_Volume 648.489 607.800 230.755  94.10 1585.60  0.909  17913
           Cloudy Traffic_Volume 665.067 624.000 234.738 

,measure,mean,median,std,min,max,skew,count
0,Traffic_Speed,61.892,62.55,10.411,19.87,90.00,-0.282,32400.0
1,Traffic_Volume,642.049,601.60,228.698,84.80,1585.60,0.914,32400.0
2,Rainfall,0.682,0.00,2.055,0.00,20.76,4.381,32400.0


Weather statistics


,Weather_Condition,measure,mean,median,std,min,max,skew,count
0,Clear,Traffic_Speed,63.015,63.830,10.029,29.21,90.00,-0.255,17913
1,Cloudy,Traffic_Speed,62.560,63.340,10.138,31.63,90.00,-0.220,6227
2,Fog,Traffic_Speed,63.625,64.505,9.768,33.35,87.11,-0.363,1614
3,Heavy Rain,Traffic_Speed,54.485,54.740,11.538,19.87,82.66,-0.189,1965
4,Light Rain,Traffic_Speed,59.221,59.300,10.162,25.05,85.64,-0.159,4681
5,Clear,Traffic_Volume,648.489,607.800,230.755,94.10,1585.60,0.909,17913
6,Cloudy,Traffic_Volume,665.067,624.000,234.738,84.80,1572.10,0.868,6227
7,Fog,Traffic_Volume,634.987,592.250,226.115,108.70,1447.50,0.929,1614
8,Heavy Rain,Traffic_Volume,606.541,575.400,211.759,101.90,1508.90,0.891,1965
9,Light Rain,Traffic_Volume,604.127,567.800,213.762,119.30,1471.20,0.959,4681


Road statistics


,Road_ID,measure,mean,median,std,min,max,skew,count
0,R01,Traffic_Speed,62.427,64.880,10.073,24.34,85.42,-0.659,2160
1,R02,Traffic_Speed,60.815,63.135,9.749,24.58,83.74,-0.642,2160
2,R03,Traffic_Speed,61.947,64.425,9.683,28.72,83.17,-0.638,2160
3,R04,Traffic_Speed,53.927,55.735,9.079,19.87,73.25,-0.630,2160
4,R05,Traffic_Speed,66.525,68.785,8.872,38.48,87.12,-0.626,2160
5,R06,Traffic_Speed,59.547,61.145,8.160,34.26,78.09,-0.575,2160
6,R07,Traffic_Speed,58.974,60.785,8.114,31.65,78.08,-0.659,2160
7,R08,Traffic_Speed,53.846,55.365,7.416,31.27,73.92,-0.594,2160
8,R09,Traffic_Speed,64.685,66.965,8.661,39.81,82.87,-0.644,2160
9,R10,Traffic_Speed,64.543,66.730,8.601,30.12,84.91,-0.665,2160


... (2 outputs left out)

Weekday statistics


,Weekday,measure,mean,median,std,min,max,skew,count
0,Friday,Traffic_Speed,60.852,61.420,10.343,19.87,86.68,-0.375,4680
1,Monday,Traffic_Speed,61.120,61.910,10.139,24.34,87.58,-0.303,4680
2,Saturday,Traffic_Speed,64.360,65.095,10.657,24.44,90.00,-0.355,4680
3,Sunday,Traffic_Speed,64.493,65.265,10.686,20.94,90.00,-0.327,4320
4,Thursday,Traffic_Speed,60.792,61.540,10.111,25.35,86.82,-0.337,4680
5,Tuesday,Traffic_Speed,60.897,61.610,10.102,24.14,90.00,-0.283,4680
6,Wednesday,Traffic_Speed,60.933,61.380,10.016,24.58,88.08,-0.224,4680
7,Friday,Traffic_Volume,693.777,650.450,225.507,223.90,1581.00,0.964,4680
8,Monday,Traffic_Volume,692.140,651.800,225.058,190.90,1538.90,0.922,4680
9,Saturday,Traffic_Volume,507.863,481.400,174.211,84.80,1257.70,0.811,4680


,Traffic_Speed,Traffic_Volume,Rainfall,hour,is_weekend
Traffic_Speed,1.000,-0.426,-0.189,-0.046,0.151
Traffic_Volume,-0.426,1.000,-0.052,0.055,-0.364
Rainfall,-0.189,-0.052,1.000,-0.034,-0.005
hour,-0.046,0.055,-0.034,1.000,0.000
is_weekend,0.151,-0.364,-0.005,0.000,1.000


,measure,lower_fence,upper_fence,outlier_count
0,Traffic_Speed,32.07,92.39,75
1,Traffic_Volume,40.80,1209.60,938
2,Rainfall,0.00,0.00,6646


## 3. Statistical comparisons and controlled effects

Normality is assessed with Shapiro-Wilk on deterministic samples (up to 5,000 per group); variance equality is checked with median-centered Levene. Weather-group outcomes use Kruskal-Wallis with Dunn-Bonferroni pairwise comparisons. Rain versus dry uses Welch tests, effect sizes, and bootstrap intervals. The controlled models adjust for road, time of day, and weekday; separate stratified results show rain/dry contrasts in each time x weekday-type cell.

In [4]:
inference = statistical_analysis(data)
print("Weather-group tests")
display(inference["group_tests"].round(5))
print("Raw versus adjusted estimates")
print(inference["raw_vs_controlled"])
print("Controlled model")
print(inference["controlled"])
print("Stratified comparisons")
display(inference["stratified"].round(4))
print("Road sensitivity ranking")
display(inference["road_sensitivity"].round(4))
print("Heterogeneity conclusion:", inference["heterogeneity"]["decision"])
print("Binary effects:", {key: value for key, value in inference["binary"].items() if "effect" in key or "cliffs" in key or "decision" in key})

Weather-group tests


,outcome,test,statistic,p_value,epsilon_squared,epsilon_magnitude,levene_p,normality_p_min,normality_by_weather
0,Traffic_Speed,Kruskal-Wallis,1320.81121,0.0,0.04065,small,0.0,0.0,"{""Clear"": 9.174208394444912e-18, ""Cloudy"": 2.6..."
1,Traffic_Volume,Kruskal-Wallis,252.61799,0.0,0.00767,small,0.0,0.0,"{""Clear"": 2.5160886711468934e-36, ""Cloudy"": 7...."


Raw versus adjusted estimates
{'raw_speed_rain_minus_dry': -5.122834256062561, 'raw_speed_rain_minus_dry_ci': [-5.388742485901468, -4.840518580181474], 'raw_speed_percent_change': -8.138813035531593, 'unadjusted_rainfall_per_mm': -0.9574033323298494, 'controlled_rainfall_per_mm': -0.9709962323637331, 'controlled_rainfall_ci': [-1.0167921759027319, -0.9252002888247344]}
Controlled model
{'formula': 'Traffic_Speed ~ Rainfall + C(Time_of_Day) + C(Weekday) + C(Road_ID)', 'rainfall_coefficient': -0.9709962323637331, 'rainfall_ci': [-1.0167921759027319, -0.9252002888247344], 'rainfall_p': 0.0, 'r_squared': 0.7835254988707744, 'interaction_formula': 'Traffic_Speed ~ Rainfall * C(Time_of_Day) + C(Weekday) + C(Road_ID)', 'interaction_coefficients': {'Rainfall': -0.826399638365087, 'Rainfall:C(Time_of_Day)[T.Midday]': -0.1820147483722736, 'Rainfall:C(Time_of_Day)[T.Morning Peak]': -0.12148586830981534, 'Rainfall:C(Time_of_Day)[T.Night]': -0.17836400720934426}, 'interaction_ci': {'Rainfall': [-0.

,Time_of_Day,weekday_type,n_rain,n_dry,mean_speed_rain,mean_speed_dry,speed_drop_pct,cohens_d_rain_minus_dry,p_value,p_fdr_bh
0,Evening Peak,Weekday,495,2430,44.7610,50.2224,10.8744,-0.8469,0.0,0.0
1,Evening Peak,Weekend,255,870,46.9534,53.1074,11.5880,-0.9212,0.0,0.0
2,Midday,Weekday,1320,4530,60.4638,65.9923,8.3775,-0.7319,0.0,0.0
3,Midday,Weekend,405,1845,63.7130,69.6758,8.5579,-0.7703,0.0,0.0
4,Morning Peak,Weekday,570,2355,45.2887,50.2188,9.8173,-0.7733,0.0,0.0
5,Morning Peak,Weekend,195,930,47.6460,53.0183,10.1330,-0.8128,0.0,0.0
6,Night,Weekday,2520,9180,60.5506,65.9995,8.2559,-0.7239,0.0,0.0
7,Night,Weekend,886,3614,64.1486,69.6879,7.9488,-0.7093,0.0,0.0


Road sensitivity ranking


,Road_ID,n_rain,n_dry,mean_speed_rain,mean_speed_dry,speed_drop_pct,speed_drop_ci_low,speed_drop_ci_high,cohens_d_rain_minus_dry,cohens_d_ci_low,cohens_d_ci_high,cohens_d_magnitude,p_value,sensitivity,p_fdr_bh,fdr_significant
0,R04,443,1717,44.8657,56.2649,20.2598,18.8157,21.6719,-1.4563,-1.5748,-1.3466,large,0.0000,High,0.0000,True
1,R01,443,1717,52.1354,65.0826,19.8935,18.4731,21.2869,-1.5034,-1.6206,-1.3864,large,0.0000,Moderate,0.0000,True
2,R02,444,1716,51.5460,63.2132,18.4570,17.0595,19.9025,-1.3671,-1.4805,-1.2552,large,0.0000,Moderate,0.0000,True
3,R03,443,1717,53.0728,64.2370,17.3798,16.0254,18.6915,-1.3026,-1.4114,-1.1927,large,0.0000,Moderate,0.0000,True
4,R05,443,1717,62.7145,67.5085,7.1013,5.7656,8.3644,-0.5536,-0.6553,-0.4475,medium,0.0000,Low,0.0000,True
5,R10,443,1717,61.0206,65.4524,6.7711,5.5244,8.1537,-0.5267,-0.6356,-0.4280,medium,0.0000,Low,0.0000,True
6,R11,443,1717,55.5487,58.7997,5.5289,4.1632,6.9717,-0.4099,-0.5193,-0.3100,small,0.0000,Low,0.0000,True
7,R06,443,1717,56.9024,60.2293,5.5237,4.1523,6.9642,-0.4133,-0.5262,-0.3106,small,0.0000,Low,0.0000,True
8,R07,443,1717,56.7946,59.5361,4.6047,3.2547,6.0264,-0.3410,-0.4485,-0.2404,small,0.0000,Low,0.0000,True
9,R12,443,1717,68.5137,71.7780,4.5477,3.2290,5.7447,-0.3604,-0.4589,-0.2579,small,0.0000,Low,0.0000,True


Heterogeneity conclusion: Evidence of heterogeneous road-level rain effects
Binary effects: {'speed_effect': {'difference': -5.122834256062561, 'difference_ci': [-5.388742485901468, -4.840518580181474], 'percent_change': -8.138813035531593, 'percent_ci': [-8.555712844884903, -7.694058984952608], 'cohens_d': -0.5020643978974136, 'd_ci': [-0.5280221891065016, -0.4743701807571832]}, 'volume_effect': {'difference': -46.81034914688894, 'difference_ci': [-52.53926382509349, -40.83002428105963], 'percent_change': -7.183345258241299, 'percent_ci': [-8.040852132729377, -6.277930519615411], 'cohens_d': -0.2053811393183076, 'd_ci': [-0.23092304376491365, -0.17907749727944974]}, 'speed_cliffs_delta': -0.26621358041878257, 'volume_cliffs_delta': -0.11741115755027585, 'speed_cliffs_ci': [-0.2921091140625, -0.2440358828125], 'volume_cliffs_ci': [-0.13952998281249998, -0.09028672031250001], 'speed_cliffs_magnitude': 'small', 'volume_cliffs_magnitude': 'small', 'speed_decision': 'Reject H0', 'volume_de

## 4. Dose response and city heterogeneity

Rainfall bins are 0, 0-2.5, 2.5-7.5, and 7.5+ mm/hour. Sensitivity labels are High at a speed drop of at least 20%, Moderate at 8% to under 20%, and Low below 8%. Road p-values are corrected with Benjamini-Hochberg FDR at alpha 0.05. The four core publication figures are written at 200 dpi.

In [5]:
print("Peak/off-peak dose-response values")
display(inference["dose"].round(3))
print("Road-by-rain interaction:", inference["heterogeneity"])
print("Sensitivity classes:")
display(inference["road_sensitivity"].groupby("sensitivity", observed=True).size().rename("roads").to_frame())

Peak/off-peak dose-response values


,rain_bin,peak_group,Traffic_Speed,Traffic_Volume
0,0,Off-peak,67.047,566.545
1,0,Peak,50.997,899.396
2,0-2.5,Off-peak,62.818,533.706
3,0-2.5,Peak,46.858,838.457
4,2.5-7.5,Off-peak,60.004,539.974
5,2.5-7.5,Peak,44.981,854.328
6,7.5+,Off-peak,58.547,531.819
7,7.5+,Peak,43.339,791.931


Road-by-rain interaction: {'f_statistic': 261.72723105980094, 'p_value': 0.0, 'road_drop_spread_percentage_points': 20.189222171826916, 'high_sensitivity_roads': ['R04'], 'fdr_significant_roads': ['R04', 'R01', 'R02', 'R03', 'R05', 'R10', 'R11', 'R06', 'R07', 'R12', 'R09', 'R13', 'R08'], 'decision': 'Evidence of heterogeneous road-level rain effects'}
Sensitivity classes:


,roads
sensitivity,
High,1
Low,11
Moderate,3


## 5. Time-aware congestion model

Congestion is speed below 60% of the road's training-period 85th-percentile speed. The chronological split uses approximately the first 75 days for training and the last 15 for testing; free-flow percentiles are calculated from training speeds only. Traffic speed is excluded from features. Traffic volume is operationally available but may be downstream of rain-related diversion, so tuned tree variants with and without it are compared. Five-fold `TimeSeriesSplit` CV tunes a pruned DecisionTreeClassifier; logistic regression and majority baselines anchor performance.

In [6]:
model_results = train_and_evaluate_models(data)
print("Model metrics")
display(model_results["metrics"].round(4))
print("Extracted decision-tree rules:")
for rule in model_results["rules"]:
    print("-", rule)
print("Confusion matrix:", model_results["confusion_matrix"])
print("Model saved to:", model_results["model_path"])

Time split: train through 2025-03-21 23:00:00, test from 2025-03-22 00:00:00
Free-flow training-only thresholds (km/h):
Road_ID
R01    72.1
R02    70.2
R03    71.5
R04    62.9
R05    74.8
R06    67.5
R07    66.7
R08    61.0
R09    72.8
R10    72.8
R11    65.8
R12    79.6
R13    79.3
R14    60.5
R15    76.0
Class balance, train: {0: 0.9818148148148148, 1: 0.018185185185185186}
Class balance, test: {0: 0.9801851851851852, 1: 0.019814814814814816}
Traffic_Volume is a legitimate operational sensor feature, but can partly mediate the weather response; the no-volume variant checks remaining rain signal.


Best tree with_volume: CV ROC-AUC=0.935 +/- 0.009; {'classifier__ccp_alpha': 0.0, 'classifier__class_weight': None, 'classifier__criterion': 'gini', 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 20, 'classifier__min_samples_split': 2}


Best tree without_volume: CV ROC-AUC=0.932 +/- 0.020; {'classifier__ccp_alpha': 0.0, 'classifier__class_weight': None, 'classifier__criterion': 'gini', 'classifier__max_depth': 8, 'classifier__min_samples_leaf': 50, 'classifier__min_samples_split': 2}


Model metrics


,model,variant,accuracy,precision,recall,f1,roc_auc,cv_roc_auc_mean,cv_roc_auc_std,best_params,train_rows,test_rows,positive_rate_test
0,Decision tree,with_volume,0.9789,0.3704,0.0935,0.1493,0.9448,0.9349,0.0095,"{""classifier__ccp_alpha"": 0.0, ""classifier__cl...",27000,5400,0.0198
1,Decision tree,without_volume,0.9802,0.0000,0.0000,0.0000,0.9447,0.9318,0.0196,"{""classifier__ccp_alpha"": 0.0, ""classifier__cl...",27000,5400,0.0198
2,Logistic regression,with_volume,0.8652,0.1143,0.8598,0.2018,0.9529,NaN,NaN,,27000,5400,0.0198
3,Majority baseline,baseline,0.9802,0.0000,0.0000,0.0000,0.5000,NaN,NaN,,27000,5400,0.0198


Extracted decision-tree rules:
- IF Rainfall <= 3.54 AND NOT Time=Morning Peak AND NOT Time=Evening Peak THEN Congested=0 (leaf support=18885, training purity=100%)
- IF Rainfall <= 3.54 AND NOT Time=Morning Peak AND Time=Evening Peak AND NOT Weather=Light Rain AND NOT Road=R04 THEN Congested=0 (leaf support=2576, training purity=98%)
- IF Rainfall <= 3.54 AND Time=Morning Peak AND Rainfall <= 0.07 AND NOT Road=R14 AND Traffic_Volume <= 995.15 THEN Congested=0 (leaf support=1754, training purity=99%)
- IF Rainfall > 3.54 AND NOT Time=Evening Peak AND NOT Time=Morning Peak AND NOT Road=R04 AND NOT Road=R02 THEN Congested=0 (leaf support=1183, training purity=98%)
- IF Rainfall <= 3.54 AND Time=Morning Peak AND Rainfall <= 0.07 AND NOT Road=R14 AND Traffic_Volume > 995.15 THEN Congested=0 (leaf support=822, training purity=96%)
Confusion matrix: [[5276, 17], [97, 10]]
Model saved to: outputs/decision_tree_model.joblib


## 6. Planner report and artifact verification

The report is populated from computed results, includes limitations and operational actions, and exports a PDF when ReportLab is available. The final cell checks the required deliverables and lists generated files.

In [7]:
report_text = write_report(data, quality, eda, inference, model_results)
required_files = [
    PROJECT_ROOT / "data/traffic_raw.csv",
    PROJECT_ROOT / "data/traffic_clean.csv",
    PROJECT_ROOT / "REPORT.md",
    PROJECT_ROOT / "REPORT.pdf",
    PROJECT_ROOT / "outputs/decision_tree_model.joblib",
    PROJECT_ROOT / "outputs/model_metrics.csv",
    PROJECT_ROOT / "outputs/road_sensitivity.csv",
    PROJECT_ROOT / "outputs/figures/01_speed_by_weather.png",
    PROJECT_ROOT / "outputs/figures/02_adjusted_speed_heatmap.png",
    PROJECT_ROOT / "outputs/figures/03_road_sensitivity.png",
    PROJECT_ROOT / "outputs/figures/04_rainfall_dose_response.png",
    PROJECT_ROOT / "outputs/figures/05_decision_tree.png",
    PROJECT_ROOT / "outputs/figures/06_feature_importance.png",
    PROJECT_ROOT / "outputs/figures/07_confusion_matrix.png",
]
missing_files = [str(path.relative_to(PROJECT_ROOT)) for path in required_files if not path.exists()]
assert not missing_files, f"Missing required deliverables: {missing_files}"
print("All required deliverables exist.")
for path in sorted(PROJECT_ROOT.rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts and ".ipynb_checkpoints" not in path.parts:
        print(path.relative_to(PROJECT_ROOT))

All required deliverables exist.
.vscode\settings.json
data\traffic_clean.csv
data\traffic_raw.csv
notebook\traffic_weather_analysis.ipynb
outputs\adjusted_weather_speed.csv
outputs\correlation_matrix.csv
outputs\data_quality_summary.csv
outputs\decision_tree_model.joblib
outputs\descriptive_overall.csv
outputs\descriptive_road.csv
outputs\descriptive_time_of_day.csv
outputs\descriptive_weather.csv
outputs\descriptive_weekday.csv
outputs\figures\01_speed_by_weather.png
outputs\figures\02_adjusted_speed_heatmap.png
outputs\figures\03_road_sensitivity.png
outputs\figures\04_rainfall_dose_response.png
outputs\figures\05_decision_tree.png
outputs\figures\06_feature_importance.png
outputs\figures\07_confusion_matrix.png
outputs\iqr_outlier_summary.csv
outputs\model_metrics.csv
outputs\posthoc_traffic_speed.csv
outputs\posthoc_traffic_volume.csv
outputs\rainfall_dose_response.csv
outputs\regression_summary.txt
outputs\road_sensitivity.csv
outputs\stratified_rain_comparisons.csv
outputs\tree_